In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A2_generation/9_4_one_hot_encoding.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Melody Generation with Machine Learning

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 21.05.2024
+ **Author:** Dr. Benedikt Zönnchen

In [ ]:
#@title install dependencies to play sound
%%capture
print('installing fluidsynth...')
!apt-get install fluidsynth > /dev/null
!cp /usr/share/sounds/sf2/FluidR3_GM.sf2 ./font.sf2
print('done!')

In [ ]:
#@title install dependencies to show score in music notation
%%capture
print('installing musescore3...')
!apt-get install musescore3 > /dev/null
print('done!')

For this notebook we need the following ``Python`` packages and moduls. Execute the cell to install them:

In [ ]:
#@title Setup: install required Python packages

%pip install pandas
%pip install numpy
%pip install torch

%pip install otter-grader==5.5.0

In [ ]:
#@title Setup: download assignment files (run this cell)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import requests, os
    folders = []
    link = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A2_generation"

    def download(entry, dest):
        if entry.get('type') != 'file' or not entry.get('download_url'):
            return
        r = requests.get(entry['download_url'])
        r.raise_for_status()
        with open(dest, 'wb') as out:
            out.write(r.content)

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        for f in requests.get(f"{link}/{folder}").json():
            download(f, f"{folder}/{f['name']}")

    for f in requests.get(link).json():
        if f['name'].endswith('.py'):
            download(f, f['name'])

    # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('9_4_one_hot_encoding.ipynb')

## 21 Representations

### 21.1 One-Hot Encoding

Let us first think about what kind of data we want to feed into the network. Consider the general case: a ``Score`` consisting of all kinds of musical objects — ``Note``s, ``Chord``s, ``Rest``s, bar lines, time signatures, and so on. A ``Score`` is a ``Stream``: a linear sequence of objects, which we will call (musical) **events**.

As we saw, *pitch* and *duration* can be represented as floating-point numbers. *Pitch* can be expressed as a MIDI note number (which may even be rational, e.g. for microtones) or as a frequency in Hz. Since a single input of a neural network is just a floating-point number, we could simply use two inputs — one for pitch and one for duration:

<img src="figs/float_input_ann.png" alt="" height="250">

This representation has a problem, however. In musical terms, two MIDI notes that are numerically close are not necessarily close in *consonance*: a minor second (1 semitone apart) is highly dissonant, while a perfect octave (12 semitones apart) is one of the most consonant intervals. The numerical proximity of MIDI values does not reflect the perceptual or harmonic relationship between notes.

For a simple melody this may not matter much, but the issue becomes more serious once we want to encode events like bar lines, time-signature changes, or ``noteOn`` / ``noteOff`` messages. And how do we encode a ``Rest``, which has no pitch at all? How is it related to a ``Note``? Such events are *categorical*, not numerical, and there is no natural ordering between them.

The **one-hot encoding** is well suited for categorical data without an inherent ordering. It is especially useful in sequence models, because it lets the network learn patterns without making unwarranted assumptions about the relative magnitudes of the inputs. But how does it work?

Imagine a vector of numbers representing the temperature along a metal rod, where each entry tells us how hot the rod is at that position. *One-hot* means that all the heat is concentrated at a single point: the vector contains exactly one ``1``, and all other entries are ``0``. The position of the ``1`` identifies the category.

Instead of treating *pitch* as a single numerical variable, we now treat it as a categorical variable with one category per pitch class:

```
C
C#
D
D#
E
...
```

A one-hot encoding of these categories then looks like this:

```
    C    C#    D    D#
    1     0    0     0     <- C
    0     1    0     0     <- C#
    0     0    1     0     <- D
    0     0    0     1     <- D#
```

Each row is the encoding of one pitch: a vector that is ``1`` at the index of the corresponding category and ``0`` everywhere else. The neural network then takes one such vector per time step as its input:

<img src="figs/one_hot_input_ann.png" alt="" height="250">

The input of the network is therefore a sequence of vectors of the form:

```
[0, 0, 1, 0, 0, ..., 0]
```

where the length of the vector equals the number of categories in our vocabulary (e.g. 12 if we use only pitch classes, or 128 if we use all MIDI pitches). Categorical events like ``Rest``, bar lines, or time-signature changes can be added to the same vocabulary simply by introducing new categories — the network treats them on equal footing with pitches, with no implicit numerical ordering between them.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

If we have 4 categories and a sequence of 3 inputs, the overall input `X` could look like this:

In [ ]:
X = [[0, 0, 1, 0], [0, 0, 1, 0], [0, 1, 0, 0]]

In essence this means that only one input node fires.

One disadvantages of this approach is that the dimensionality of the input can explode. For ``128`` MIDI notes (without considering duration) we would require ``128``-dimensional vectors.

The following code generates a ``DataFrame`` of one hot-encoded colors, where each row represents one color. `True` means `1` and `False` means `0`.

In [ ]:
# Sample data
data = {'Color': ['Red', 'Blue', 'Green', 'Yellow', 'Red', 'Red' ]}
df = pd.DataFrame(data)

# One-hot encoding
one_hot_encoded_df = pd.get_dummies(df, columns=['Color'])

print(one_hot_encoded_df)

In ``NumPy`` ``tf.one_hot(input, depth=num_classes)`` does the trick but here we have to privide a ``numpy`` array of natural numbers ranging from ``0`` to ``depth-1``. This basically means that we have to assign consecutive natural numbers starting from ``0`` to our categories beforehand.

In [ ]:
X = torch.tensor([0, 0, 1, 3, 1, 6])

X_one_hot = F.one_hot(X, num_classes=7).float()
X_one_hot

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 21.1:** Explain how a *one-hot* encoding of the following sequence of **words** might look like.


``I`` ``like`` ``to`` ``eat`` ``and`` ``I`` ``like`` ``to`` ``play``

How many dimensions are required if you only want to encode this sequence?

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

